# Experiments Analysis

Retrieve and filter LangSmith evaluation runs for flexible post-hoc analysis.

In [1]:
import sys
import os
import json
from collections import Counter

sys.path.insert(0, os.path.abspath(".."))

from typing import Any, Callable, Dict, List, Optional

from langsmith import Client

client = Client()

In [2]:
def load_experiment_runs(
    experiment_prefix: str = "",
    dataset_name: str = "Dataset v1.",
) -> List[Dict[str, Any]]:
    """
    Retrieve all evaluation runs from LangSmith experiments and return them
    as a flat list of dicts ready for filtering and inspection.

    Each record contains:
        experiment         — experiment (project) name
        task_id            — the eval task identifier
        thread_id          — the pipeline checkpoint thread id
        run_id             — LangSmith run id
        scores             — dict of {score_key: float} from all evaluators
        <score_key>        — every score also flattened to top level for easy filtering
        final_code         — generated code from the pipeline state
        retrieval_context  — RAG context from the pipeline state
        comments           — dict of {feedback_key: comment_string} from evaluators
        statements         — parsed JSON from first available comment (or None if no valid JSON)
        outputs            — raw LangSmith run outputs (full pre_computed_state inside)

    Args:
        experiment_prefix: Only include experiments whose name starts with this string.
                           Empty string means include all experiments for the dataset.
        dataset_name:      LangSmith dataset name used to scope experiments.
    """
    records: List[Dict[str, Any]] = []

    projects = list(client.list_projects(reference_dataset_name=dataset_name))
    if experiment_prefix:
        projects = [p for p in projects if p.name.startswith(experiment_prefix)]

    print(f"Found {len(projects)} experiment(s) matching prefix '{experiment_prefix}'")

    for project in projects:
        runs = list(client.list_runs(project_name=project.name, is_root=True))

        if not runs:
            continue

        # Bulk-fetch all feedback for this project in one call
        run_ids = [str(r.id) for r in runs]
        feedback_by_run: Dict[str, Dict[str, Any]] = {rid: {} for rid in run_ids}
        for fb in client.list_feedback(run_ids=run_ids):
            rid = str(fb.run_id)

            if fb.score is not None:
                feedback_by_run[rid][fb.key] = fb.score
            
            # Store the comment for each feedback key as well
            if fb.comment:
                if f"{fb.key}_comment" not in feedback_by_run[rid]:
                    feedback_by_run[rid][f"{fb.key}_comment"] = fb.comment

        for run in runs:
            rid = str(run.id)
            feedback_data = feedback_by_run.get(rid, {})
            scores = {k: v for k, v in feedback_data.items() if not k.endswith("_comment")}
            outputs = run.outputs or {}
            pre_state = outputs.get("pre_computed_state") or {}

            task_id = (
                str(outputs.get("task_id") or "").strip()
                or str(pre_state.get("task_id") or "").strip()
            )

            # Extract and parse statements from feedback comments
            # Try to parse as JSON; if it fails, keep the raw comment
            statements_parsed = None
            comments_dict = {}
            
            for key, value in feedback_data.items():
                if key.endswith("_comment"):
                    comment_key = key.replace("_comment", "")
                    
                    # Try to parse as JSON
                    if value:
                        try:
                            parsed = json.loads(value)
                            comments_dict[comment_key] = parsed
                            if statements_parsed is None:
                                statements_parsed = parsed
                        except (json.JSONDecodeError, ValueError):
                            # If JSON parsing fails, keep raw comment string
                            comments_dict[comment_key] = value
                    else:
                        comments_dict[comment_key] = value

            record: Dict[str, Any] = {
                "experiment": project.name,
                "task_id": task_id,
                "thread_id": outputs.get("thread_id", ""),
                "run_id": rid,
                "scores": scores,
                "final_code": pre_state.get("final_code") or pre_state.get("generated_code") or "",
                "retrieval_context": pre_state.get("retrieval_context", ""),
                "comments": comments_dict,
                "statements": statements_parsed,
                "outputs": outputs,
            }
            # Flatten scores to top level for convenient filtering
            record.update(scores)

            records.append(record)

    print(f"Loaded {len(records)} total run(s)")
    return records

In [3]:
def filter_runs(
    records: List[Dict[str, Any]],
    **conditions: Any,
) -> List[Dict[str, Any]]:
    """
    Filter records by field values.

    Each keyword argument is a field name. The value can be:
    - A scalar     → exact equality match  (e.g., code_execution_score=0.0)
    - A callable   → predicate on the field (e.g., task_id=lambda v: "add" in v)

    Missing fields evaluate to None against the condition.

    Example:
        filter_runs(records, code_statements_score=1.0, code_execution_score=0.0)
        filter_runs(records, task_id=lambda v: v.startswith("add"))
    """
    result = []
    for record in records:
        if all(
            (cond(record.get(key)) if callable(cond) else record.get(key) == cond)
            for key, cond in conditions.items()
        ):
            result.append(record)
    return result


def display_runs(
    records: List[Dict[str, Any]],
    show_code: bool = False,
    show_context: bool = False,
    code_preview_chars: int = 600,
    context_preview_chars: int = 400,
) -> None:
    """
    Pretty-print a list of run records.

    Args:
        records:              Output of load_experiment_runs / filter_runs.
        show_code:            Print a preview of final_code.
        show_context:         Print a preview of retrieval_context.
        code_preview_chars:   Max chars to show for code preview.
        context_preview_chars: Max chars to show for context preview.
    """
    SEP = "─" * 70
    print(f"{len(records)} record(s)\n{SEP}")

    for i, r in enumerate(records, 1):
        scores = r.get("scores", {})
        score_str = "  ".join(
            f"{k}={v:.2f}" for k, v in sorted(scores.items()) if v is not None
        )
        print(f"\n[{i}]  task_id={r.get('task_id', '?')!r}")
        print(f"     experiment: {r.get('experiment', '?')}")
        print(f"     thread_id:  {r.get('thread_id', '?')}")
        print(f"     scores:     {score_str or '(none)'}")

        if show_code:
            code = r.get("final_code", "") or ""
            snippet = code[:code_preview_chars]
            ellipsis = "..." if len(code) > code_preview_chars else ""
            print(f"     final_code:\n{snippet}{ellipsis}")

        if show_context:
            ctx = r.get("retrieval_context", "") or ""
            snippet = ctx[:context_preview_chars]
            ellipsis = "..." if len(ctx) > context_preview_chars else ""
            print(f"     retrieval_context:\n{snippet}{ellipsis}")

    print(f"\n{SEP}")

## Usage Examples

In [7]:
# --- Load ---
# Empty prefix loads all experiments for the dataset.
# Supply a prefix string to scope to a specific experiment family.
EXPERIMENT_PREFIX = "COMPLEX"       # e.g. "2) With concision"
DATASET_NAME = "Dataset v2."

records = load_experiment_runs(
    experiment_prefix=EXPERIMENT_PREFIX,
    dataset_name=DATASET_NAME,
)

Found 3 experiment(s) matching prefix 'COMPLEX'
Loaded 15 total run(s)


In [8]:

# --- Verify JSON parsing is integrated into comments ---
print("=== Verifying integrated JSON parsing ===\n")

r = records[0]
print(f"Task: {r['task_id']}\n")

for comment_type, comment_value in r['comments'].items():
    print(f"[{comment_type}]")
    print(f"  Type: {type(comment_value).__name__}")
    
    if isinstance(comment_value, dict):
        print(f"  Dict keys: {list(comment_value.keys())}")
        if 'pass' in comment_value:
            print(f"  Pass: {comment_value['pass']}")
    elif isinstance(comment_value, list):
        print(f"  List length: {len(comment_value)}")
        if comment_value and isinstance(comment_value[0], dict):
            print(f"  First item keys: {list(comment_value[0].keys())}")
    else:
        print(f"  Raw string (parsing failed): {comment_value[:100]}")
    print()

# Count how many comments are parsed vs raw
parsed_count = sum(1 for r in records for c in r['comments'].values() if not isinstance(c, str))
raw_count = sum(1 for r in records for c in r['comments'].values() if isinstance(c, str))
total_comments = sum(len(r['comments']) for r in records)

print(f"Summary:")
print(f"  Total comments: {total_comments}")
print(f"  Parsed (JSON): {parsed_count}")
print(f"  Raw (string): {raw_count}")


=== Verifying integrated JSON parsing ===

Task: archive_stale_tasks

[code_execution_score]
  Type: dict
  Dict keys: ['pass', 'output', 'errors']
  Pass: True

[awrapper]
  Type: str
  Raw string (parsing failed): ValueError('Failed to extract JSON from response. Content preview: ```json\n[\n  {\n    "statement":

[rag_statements_score]
  Type: list
  List length: 6
  First item keys: ['statement', 'status', 'evidence', 'reasoning']

Summary:
  Total comments: 45
  Parsed (JSON): 44
  Raw (string): 1


## Code & RAG Statement Failure Analysis

Analyze CODE and RAG statement failures with proper status/reasoning structure.


In [26]:
def extract_statement_failures(statements_list):
    """
    Parse failures from statement list with proper status/reasoning structure.
    
    Statement format:
      {
        "statement": str,        # the claim being checked
        "status": str,           # "Present" | "Wrong" | "Not present"
        "evidence": str,         # verbatim quote from context
        "reasoning": str         # explanation of the status
      }
    
    A statement FAILS when status is "Wrong" or "Not present".
    Returns dict mapping failure_key -> {count, status_type, reasoning, examples}
    """
    failures = {}
    
    if not isinstance(statements_list, list):
        return failures
    
    for stmt in statements_list:
        if not isinstance(stmt, dict):
            continue
        
        status = stmt.get("status", "").strip()
        if status not in ("Wrong", "Not present"):
            continue  # Only track failures
        
        statement_text = stmt.get("statement", "Unknown statement")[:100]
        reasoning = stmt.get("reasoning", "No explanation provided")
        evidence = stmt.get("evidence", "NONE")
        
        # Composite key: status + reasoning
        failure_key = f"{status}: {reasoning}"
        
        if failure_key not in failures:
            failures[failure_key] = {
                "count": 0,
                "status_type": status,
                "reasoning": reasoning,
                "examples": []
            }
        
        failures[failure_key]["count"] += 1
        if len(failures[failure_key]["examples"]) < 2:
            failures[failure_key]["examples"].append({
                "statement": statement_text,
                "evidence": evidence[:200]
            })
    
    return failures


print("✅ Statement failure extractor loaded.")


✅ Statement failure extractor loaded.


In [27]:
def analyze_code_failures(records):
    """
    Analyze CODE statement failures by task and failure type.
    Returns: (failures_by_task, case_samples)
    """
    from collections import defaultdict
    
    failures_by_task = defaultdict(dict)
    case_samples = defaultdict(lambda: defaultdict(list))
    
    for record in records:
        comments = record.get("comments", {}) or {}
        code_stmts = comments.get("code_statements_score")
        task_id = record.get("task_id", "unknown")
        run_id = record.get("run_id", "")
        
        if not isinstance(code_stmts, list):
            continue
        
        record_failures = extract_statement_failures(code_stmts)
        for failure_key, failure_info in record_failures.items():
            if failure_key not in failures_by_task[task_id]:
                failures_by_task[task_id][failure_key] = {
                    "count": 0,
                    "status_type": failure_info["status_type"],
                    "reasoning": failure_info["reasoning"]
                }
            
            failures_by_task[task_id][failure_key]["count"] += failure_info["count"]
            case_samples[task_id][failure_key].append({
                "run_id": run_id,
                "examples": failure_info["examples"],
                "code": record.get("final_code", "")[:300]
            })
    
    return failures_by_task, case_samples


code_failures_by_task, code_case_samples = analyze_code_failures(records)

print("\n" + "="*90)
print("CODE STATEMENT FAILURES - BY TASK AND FAILURE TYPE")
print("="*90)

for task_id in sorted(code_failures_by_task.keys()):
    failures = code_failures_by_task[task_id]
    total = sum(f["count"] for f in failures.values())
    
    print(f"\n📋 TASK: {task_id}")
    print(f"   Total failures: {total} | Failure types: {len(failures)}")
    print(f"   {'-'*82}")
    
    for failure_key in sorted(failures.keys(), key=lambda k: failures[k]["count"], reverse=True):
        info = failures[failure_key]
        pct = (info["count"] / total * 100) if total > 0 else 0
        
        print(f"\n   [{info['status_type']}] {info['reasoning']}")
        print(f"        Count: {info['count']} ({pct:.1f}%)")
        
        # Show example statement + evidence
        samples = code_case_samples[task_id][failure_key]
        if samples and samples[0]["examples"]:
            ex = samples[0]["examples"][0]
            print(f"        Example statement: {ex['statement']}")
            print(f"        Evidence: {ex['evidence']}")

print(f"\n{'='*90}\n")



CODE STATEMENT FAILURES - BY TASK AND FAILURE TYPE

📋 TASK: archive_stale_tasks
   Total failures: 6 | Failure types: 4
   ----------------------------------------------------------------------------------

   [Not present] The code does not use the specified query endpoint or payload.
        Count: 2 (33.3%)
        Example statement: The stale filter uses POST https://api.notion.com/v1/databases/{database_id}/query with {'property':
        Evidence: NONE

   [Not present] The code does not use a PATCH request to archive tasks.
        Count: 2 (33.3%)
        Example statement: For every matched page id, PATCH https://api.notion.com/v1/pages/{page_id} is called with top-level 
        Evidence: NONE

   [Wrong] The 'rich_text' structure is different; missing 'type': 'text'.
        Count: 1 (16.7%)
        Example statement: Comment payload uses {'parent': {'page_id': '<page_id>'}, 'rich_text': [{'type': 'text', 'text': {'c
        Evidence: payload = {
                "parent": {

In [28]:
def analyze_rag_failures(records):
    """
    Analyze RAG statement failures by task and failure type.
    Returns: (failures_by_task, case_samples)
    """
    from collections import defaultdict
    
    failures_by_task = defaultdict(dict)
    case_samples = defaultdict(lambda: defaultdict(list))
    
    for record in records:
        comments = record.get("comments", {}) or {}
        rag_stmts = comments.get("rag_statements_score")
        task_id = record.get("task_id", "unknown")
        run_id = record.get("run_id", "")
        
        if not isinstance(rag_stmts, list):
            continue
        
        record_failures = extract_statement_failures(rag_stmts)
        for failure_key, failure_info in record_failures.items():
            if failure_key not in failures_by_task[task_id]:
                failures_by_task[task_id][failure_key] = {
                    "count": 0,
                    "status_type": failure_info["status_type"],
                    "reasoning": failure_info["reasoning"]
                }
            
            failures_by_task[task_id][failure_key]["count"] += failure_info["count"]
            case_samples[task_id][failure_key].append({
                "run_id": run_id,
                "examples": failure_info["examples"],
                "context": record.get("retrieval_context", "")[:300]
            })
    
    return failures_by_task, case_samples


rag_failures_by_task, rag_case_samples = analyze_rag_failures(records)

print("\n" + "="*90)
print("RAG STATEMENT FAILURES - BY TASK AND FAILURE TYPE")
print("="*90)

for task_id in sorted(rag_failures_by_task.keys()):
    failures = rag_failures_by_task[task_id]
    total = sum(f["count"] for f in failures.values())
    
    print(f"\n📋 TASK: {task_id}")
    print(f"   Total failures: {total} | Failure types: {len(failures)}")
    print(f"   {'-'*82}")
    
    for failure_key in sorted(failures.keys(), key=lambda k: failures[k]["count"], reverse=True):
        info = failures[failure_key]
        pct = (info["count"] / total * 100) if total > 0 else 0
        
        print(f"\n   [{info['status_type']}] {info['reasoning']}")
        print(f"        Count: {info['count']} ({pct:.1f}%)")
        
        # Show example statement + evidence
        samples = rag_case_samples[task_id][failure_key]
        if samples and samples[0]["examples"]:
            ex = samples[0]["examples"][0]
            print(f"        Example statement: {ex['statement']}")
            print(f"        Evidence: {ex['evidence']}")

print(f"\n{'='*90}\n")



RAG STATEMENT FAILURES - BY TASK AND FAILURE TYPE

📋 TASK: archive_stale_tasks
   Total failures: 9 | Failure types: 6
   ----------------------------------------------------------------------------------

   [Not present] The context does not mention the `Notion-Version` header.
        Count: 2 (22.2%)
        Example statement: Required Headers: `Notion-Version` (value not specified in context)
        Evidence: NONE

   [Not present] The context does not mention capability requirements.
        Count: 2 (22.2%)
        Example statement: Integration must have "insert comment" capability enabled.
        Evidence: NONE

   [Not present] The context only states a threshold date is calculated.
        Count: 2 (22.2%)
        Example statement: Technical Statements: Python datetime arithmetic computes a threshold date as current date minus tim
        Evidence: NONE

   [Not present] Context does not detail implementation specifics like Python code.
        Count: 1 (11.1%)
        E

In [29]:
def comprehensive_statement_analysis(records):
    """
    Compare CODE vs RAG failures side-by-side for all tasks.
    Shows which failure reasons appear in both and their impact.
    """
    from collections import Counter
    
    analysis = {}
    
    for record in records:
        comments = record.get("comments", {}) or {}
        task_id = record.get("task_id", "unknown")
        
        if task_id not in analysis:
            analysis[task_id] = {
                "total_runs": 0,
                "code_failures": 0,
                "rag_failures": 0,
                "both_failed": 0,
                "code_only": 0,
                "rag_only": 0,
                "code_failure_reasons": Counter(),
                "rag_failure_reasons": Counter(),
            }
        
        analysis[task_id]["total_runs"] += 1
        
        # Extract code failures
        code_stmts = comments.get("code_statements_score")
        code_has_failures = False
        if isinstance(code_stmts, list):
            code_failures = extract_statement_failures(code_stmts)
            if code_failures:
                code_has_failures = True
                for key, info in code_failures.items():
                    analysis[task_id]["code_failure_reasons"][f"[{info['status_type']}] {info['reasoning']}"] += info["count"]
        
        # Extract RAG failures
        rag_stmts = comments.get("rag_statements_score")
        rag_has_failures = False
        if isinstance(rag_stmts, list):
            rag_failures = extract_statement_failures(rag_stmts)
            if rag_failures:
                rag_has_failures = True
                for key, info in rag_failures.items():
                    analysis[task_id]["rag_failure_reasons"][f"[{info['status_type']}] {info['reasoning']}"] += info["count"]
        
        # Update counters
        if code_has_failures:
            analysis[task_id]["code_failures"] += 1
        if rag_has_failures:
            analysis[task_id]["rag_failures"] += 1
        if code_has_failures and rag_has_failures:
            analysis[task_id]["both_failed"] += 1
        elif code_has_failures:
            analysis[task_id]["code_only"] += 1
        elif rag_has_failures:
            analysis[task_id]["rag_only"] += 1
    
    return analysis


task_analysis = comprehensive_statement_analysis(records)

print("\n" + "="*90)
print("COMPREHENSIVE FAILURE COMPARISON - CODE VS RAG BY TASK")
print("="*90)

for task_id in sorted(task_analysis.keys()):
    info = task_analysis[task_id]
    total = info["total_runs"]
    code_pct = (info["code_failures"] / total * 100) if total > 0 else 0
    rag_pct = (info["rag_failures"] / total * 100) if total > 0 else 0
    
    print(f"\n{'═'*90}")
    print(f"📌 TASK: {task_id}")
    print(f"{'═'*90}")
    print(f"  Total runs: {total}")
    print(f"  Code failures: {info['code_failures']} ({code_pct:.1f}%)")
    print(f"  RAG failures:  {info['rag_failures']} ({rag_pct:.1f}%)")
    print(f"\n  Overlap:")
    print(f"    • Both failed:  {info['both_failed']}")
    print(f"    • Code only:    {info['code_only']}")
    print(f"    • RAG only:     {info['rag_only']}")
    
    # Top CODE failures
    if info["code_failure_reasons"]:
        print(f"\n  Top CODE failure reasons:")
        for reason, count in info["code_failure_reasons"].most_common(3):
            print(f"    • {reason}")
            print(f"      Count: {count}")
    else:
        print(f"\n  Code failures: NONE")
    
    # Top RAG failures
    if info["rag_failure_reasons"]:
        print(f"\n  Top RAG failure reasons:")
        for reason, count in info["rag_failure_reasons"].most_common(3):
            print(f"    • {reason}")
            print(f"      Count: {count}")
    else:
        print(f"\n  RAG failures: NONE")

print(f"\n{'═'*90}\n")



COMPREHENSIVE FAILURE COMPARISON - CODE VS RAG BY TASK

══════════════════════════════════════════════════════════════════════════════════════════
📌 TASK: archive_stale_tasks
══════════════════════════════════════════════════════════════════════════════════════════
  Total runs: 3
  Code failures: 2 (66.7%)
  RAG failures:  3 (100.0%)

  Overlap:
    • Both failed:  2
    • Code only:    0
    • RAG only:     1

  Top CODE failure reasons:
    • [Not present] The code does not use the specified query endpoint or payload.
      Count: 2
    • [Not present] The code does not use a PATCH request to archive tasks.
      Count: 2
    • [Wrong] The 'rich_text' structure is different; missing 'type': 'text'.
      Count: 1

  Top RAG failure reasons:
    • [Not present] The context does not mention the `Notion-Version` header.
      Count: 2
    • [Not present] The context does not mention capability requirements.
      Count: 2
    • [Not present] The context only states a threshold date is

In [30]:
def inspect_statement_failures(records, task_id, statement_type="code", max_samples=10):
    """
    Drill down into specific statement failures for detailed inspection.
    
    Args:
        records: List of run records
        task_id: Filter by this task
        statement_type: "code" or "rag"
        max_samples: Max failure cases to show
    
    Returns structured failure cases with full details.
    """
    from collections import defaultdict
    
    score_key = f"{statement_type}_statements_score"
    failure_cases = defaultdict(list)
    
    for record in records:
        if record.get("task_id") != task_id:
            continue
        
        comments = record.get("comments", {}) or {}
        stmts = comments.get(score_key)
        if not isinstance(stmts, list):
            continue
        
        stmt_failures = extract_statement_failures(stmts)
        if not stmt_failures:
            continue
        
        for failure_key, failure_info in stmt_failures.items():
            for example in failure_info["examples"]:
                failure_cases[failure_key].append({
                    "run_id": record.get("run_id", ""),
                    "thread_id": record.get("thread_id", ""),
                    "statement": example["statement"],
                    "evidence": example["evidence"],
                    "code": record.get("final_code", "") if statement_type == "code" else None,
                    "context": record.get("retrieval_context", "") if statement_type == "rag" else None,
                })
    
    return failure_cases


def show_statement_failures(records, task_id, statement_type="code"):
    """
    Pretty-print detailed statement failure cases.
    """
    cases = inspect_statement_failures(records, task_id, statement_type)
    
    print(f"\n{'='*100}")
    print(f"DETAILED {statement_type.upper()} STATEMENT FAILURES: Task={task_id}")
    print(f"{'='*100}\n")
    
    for failure_key in sorted(cases.keys()):
        case_list = cases[failure_key]
        print(f"{'─'*100}")
        print(f"❌ {failure_key}")
        print(f"   Total occurrences: {len(case_list)}")
        print(f"{'─'*100}\n")
        
        # Show first 3 detailed cases
        for i, case in enumerate(case_list[:3], 1):
            print(f"   Case {i}: Run {case['run_id']}")
            print(f"   ├─ Statement: {case['statement']}")
            print(f"   ├─ Evidence: {case['evidence']}")
            
            if statement_type == "code" and case["code"]:
                print(f"   └─ Code (first 500 chars):\n       {case['code'][:500]}")
            elif statement_type == "rag" and case["context"]:
                print(f"   └─ Context (first 500 chars):\n       {case['context'][:500]}")
            print()

print("\n✅ Failure inspection tools ready!")
print("\nUsage examples:")
print("  show_statement_failures(records, task_id='add_task', statement_type='code')")
print("  show_statement_failures(records, task_id='add_task', statement_type='rag')")



✅ Failure inspection tools ready!

Usage examples:
  show_statement_failures(records, task_id='add_task', statement_type='code')
  show_statement_failures(records, task_id='add_task', statement_type='rag')


In [31]:
print("\n" + "="*100)
print("STATEMENT FAILURE ANALYSIS SUMMARY")
print("="*100)

# Summary stats
code_tasks_with_failures = sum(1 for t, f in code_failures_by_task.items() if f)
rag_tasks_with_failures = sum(1 for t, f in rag_failures_by_task.items() if f)
total_code_failures = sum(sum(f[k]["count"] for k in f) for f in code_failures_by_task.values())
total_rag_failures = sum(sum(f[k]["count"] for k in f) for f in rag_failures_by_task.values())

print(f"\n📊 OVERVIEW:")
print(f"   Tasks analyzed:           {len(records)}")
print(f"   Tasks with CODE failures: {code_tasks_with_failures}")
print(f"   Tasks with RAG failures:  {rag_tasks_with_failures}")
print(f"   Total CODE failures:      {total_code_failures}")
print(f"   Total RAG failures:       {total_rag_failures}")

print(f"\n📋 ANALYSIS FUNCTIONS:")
print(f"   1. analyze_code_failures(records)")
print(f"      → Breakdown of CODE statement failures by task and reason")
print(f"      → Returns: (failures_by_task, case_samples)")
print(f"\n   2. analyze_rag_failures(records)")
print(f"      → Breakdown of RAG statement failures by task and reason")
print(f"      → Returns: (failures_by_task, case_samples)")
print(f"\n   3. comprehensive_statement_analysis(records)")
print(f"      → Compare CODE vs RAG failures side-by-side")
print(f"      → Returns: dict with both/code_only/rag_only counts")
print(f"\n   4. show_statement_failures(records, task_id, statement_type)")
print(f"      → Drill into specific failures with full evidence")
print(f"      → statement_type: 'code' or 'rag'")

print(f"\n🔍 QUICK LOOKUP:")
print(f"\n   Task Name             | Code Failures | RAG Failures")
print(f"   {'-'*60}")
for task_id in sorted(task_analysis.keys()):
    code_count = task_analysis[task_id]["code_failures"]
    rag_count = task_analysis[task_id]["rag_failures"]
    print(f"   {task_id:20} | {code_count:13} | {rag_count:12}")

print(f"\n" + "="*100 + "\n")



STATEMENT FAILURE ANALYSIS SUMMARY

📊 OVERVIEW:
   Tasks analyzed:           15
   Tasks with CODE failures: 5
   Tasks with RAG failures:  5
   Total CODE failures:      40
   Total RAG failures:       53

📋 ANALYSIS FUNCTIONS:
   1. analyze_code_failures(records)
      → Breakdown of CODE statement failures by task and reason
      → Returns: (failures_by_task, case_samples)

   2. analyze_rag_failures(records)
      → Breakdown of RAG statement failures by task and reason
      → Returns: (failures_by_task, case_samples)

   3. comprehensive_statement_analysis(records)
      → Compare CODE vs RAG failures side-by-side
      → Returns: dict with both/code_only/rag_only counts

   4. show_statement_failures(records, task_id, statement_type)
      → Drill into specific failures with full evidence
      → statement_type: 'code' or 'rag'

🔍 QUICK LOOKUP:

   Task Name             | Code Failures | RAG Failures
   ------------------------------------------------------------
   archive_st